# SEMANA 4: INTEGRACIÓN Y LIMPIEZA. TABLA DE VARIABLES Y CREACIÓN DE NUEVAS VARIABLES. TRANSFORMACIÓN DE DATOS

### Integrante:
### Anccasi Pozo Joseph 

### 1.	Con base a datos de cáncer de mama alojados en el repositorio UCI Machine Learning Repository  y utilizando las librerías de Python que se indican, haga lo siguiente:

#### a. Por medio de la librería ‘Pandas’, lea la base de datos, asigne nombres para las columnas, realice una imputación para la variable ‘Bare Nuclei’ por medio de la mediana y cambie las categorías para la variable ‘Class’ de 2 o 4 por 0 o 1

In [41]:
import pandas as pd
import numpy as np


In [43]:
# Nombres de las columnas según el sitio de UCI
columnas = [
    'Sample code number', 'Clump Thickness', 'Uniformity of Cell Size', 'Uniformity of Cell Shape',
    'Marginal Adhesion', 'Single Epithelial Cell Size', 'Bare Nuclei',
    'Bland Chromatin', 'Normal Nucleoli', 'Mitoses', 'Class'
]

# Leer los datos desde la URL
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"
df = pd.read_csv(url, names=columnas)
df

,Sample code number,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1,3,1,1,2
1,1002945,5,4,4,5,7,10,3,2,1,2
2,1015425,3,1,1,1,2,2,3,1,1,2
3,1016277,6,8,8,1,3,4,3,7,1,2
4,1017023,4,1,1,3,2,1,3,1,1,2
...,...,...,...,...,...,...,...,...,...,...,...
694,776715,3,1,1,1,3,2,1,1,1,2
695,841769,2,1,1,1,2,1,1,1,1,2
696,888820,5,10,10,3,7,3,8,10,2,4
697,897471,4,8,6,4,3,4,10,6,1,4


In [45]:
# Reemplazamos los valores faltantes representados con '?' por NaN
df.replace("?", np.nan, inplace=True)

In [47]:
# Convertimos la columna a numérica para poder trabajar con ella (ya que puede tener valores no numéricos como 'n/a')
df['Bare Nuclei'] = pd.to_numeric(df['Bare Nuclei'], errors='coerce')

In [49]:
# Imputamos los valores faltantes en 'Bare_Nuclei' con la mediana
median_bare_nuclei = df["Bare Nuclei"].median()
df["Bare Nuclei"].fillna(median_bare_nuclei, inplace=True)

In [51]:
# Transformamos la variable 'Class': 2 (benigno) -> 0, 4 (maligno) -> 1
df["Class"] = df["Class"].map({2: 0, 4: 1})

In [53]:
# Mostramos el  DataFrame limpio
df

,Sample code number,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...
694,776715,3,1,1,1,3,2.0,1,1,1,0
695,841769,2,1,1,1,2,1.0,1,1,1,0
696,888820,5,10,10,3,7,3.0,8,10,2,1
697,897471,4,8,6,4,3,4.0,10,6,1,1


#### b.	Realice una detección de valores atípicos univariados por medio del método del rango intercuartílico con 3 de longitud a la derecha y 3 a la izquierda, y elimínelos. Además, realice una detección de valores atípicos multivariados por medio de las distancias de Mahalanobis y elimine aquellos valores que superen el valor de 30.

In [55]:
import numpy as np

# Copia del DataFrame original para no modificarlo accidentalmente
df_clean = df.copy()

# Eliminamos la columna de ID porque no aporta valor en el análisis
df_clean.drop(columns=['Sample code number'], inplace=True)

# Outliers univariados usando IQR con rango 3
for col in df_clean.columns[:-1]:  # Excluimos 'Class'
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

In [57]:
from scipy.spatial import distance
import numpy as np

# Matriz de características (sin la clase)
X = df_clean.drop(columns='Class')

# Calcular la media y la matriz de covarianza
mean_vec = np.mean(X, axis=0)
cov_matrix = np.cov(X, rowvar=False)

# Usar pseudoinversa para evitar el error de matriz singular
inv_cov_matrix = np.linalg.pinv(cov_matrix)

# Calcular distancia de Mahalanobis
mahal_distances = X.apply(lambda row: distance.mahalanobis(row, mean_vec, inv_cov_matrix), axis=1)

# Agregar la distancia al DataFrame
df_clean['Mahalanobis'] = mahal_distances

# Eliminar outliers con distancia > 30
df_clean = df_clean[df_clean['Mahalanobis'] <= 30]

# Quitar la columna auxiliar
df_clean.drop(columns='Mahalanobis', inplace=True)

# Resetear índice
df_clean.reset_index(drop=True, inplace=True)

# Mostrar el dataframe
df_clean


,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,5,1,1,1,2,1.0,3,1,1,0
1,5,4,4,5,7,10.0,3,2,1,0
2,3,1,1,1,2,2.0,3,1,1,0
3,6,8,8,1,3,4.0,3,7,1,0
4,4,1,1,3,2,1.0,3,1,1,0
...,...,...,...,...,...,...,...,...,...,...
574,3,1,1,1,2,1.0,1,1,1,0
575,3,1,1,1,3,2.0,1,1,1,0
576,2,1,1,1,2,1.0,1,1,1,0
577,4,8,6,4,3,4.0,10,6,1,1


#### c.	Realice un cálculo de los principales estadísticos descriptivos para las variables predictoras y cree dos conjuntos de datos: uno en donde se aplique la transformación Z-score y otro en donde se aplique la normalización mín-máx, sobre todas las variables predictoras.

In [59]:
# Variables predictoras (sin la clase)
X = df_clean.drop(columns='Class')

# Estadísticos descriptivos
desc_stats = X.describe()
desc_stats


,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses
count,579.000000,579.000000,579.000000,579.000000,579.00000,579.000000,579.000000,579.000000,579.0
mean,3.851468,2.449050,2.578584,2.219344,2.75475,2.718480,3.003454,2.184801,1.0
std,2.528463,2.587315,2.550926,2.315565,1.80789,3.153268,2.199400,2.491239,0.0
min,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.0
25%,2.000000,1.000000,1.000000,1.000000,2.00000,1.000000,1.500000,1.000000,1.0
50%,3.000000,1.000000,1.000000,1.000000,2.00000,1.000000,2.000000,1.000000,1.0
75%,5.000000,3.000000,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,1.0
max,10.000000,10.000000,10.000000,10.000000,10.00000,10.000000,10.000000,10.000000,1.0


In [61]:
from sklearn.preprocessing import StandardScaler

# Inicializar el escalador
scaler_z = StandardScaler()

# Aplicar la transformación
X_zscore = scaler_z.fit_transform(X)

# Crear nuevo DataFrame con Z-score
df_zscore = pd.DataFrame(X_zscore, columns=X.columns)

# Opcional: agregar la columna 'Class' si deseas conservarla
df_zscore['Class'] = df_clean['Class'].values

# Ver los primeros registros
df_zscore.head()


,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,0.454634,-0.560544,-0.619363,-0.527041,-0.417836,-0.545455,-0.001572,-0.475998,0.0,0
1,0.454634,0.599962,0.557698,1.201893,2.350209,2.311194,-0.001572,-0.074245,0.0,0
2,-0.337044,-0.560544,-0.619363,-0.527041,-0.417836,-0.228050,-0.001572,-0.475998,0.0,0
3,0.850473,2.147303,2.127112,-0.527041,0.135773,0.406761,-0.001572,1.934524,0.0,0
4,0.058795,-0.560544,-0.619363,0.337426,-0.417836,-0.545455,-0.001572,-0.475998,0.0,0


In [63]:
from sklearn.preprocessing import MinMaxScaler

# Inicializar el escalador
scaler_minmax = MinMaxScaler()

# Aplicar la transformación
X_minmax = scaler_minmax.fit_transform(X)

# Crear nuevo DataFrame con normalización Min-Max
df_minmax = pd.DataFrame(X_minmax, columns=X.columns)

# Agregar columna 'Class'
df_minmax['Class'] = df_clean['Class'].values

# Ver los primeros registros
df_minmax.head()


,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,0.444444,0.000000,0.000000,0.000000,0.111111,0.000000,0.222222,0.000000,0.0,0
1,0.444444,0.333333,0.333333,0.444444,0.666667,1.000000,0.222222,0.111111,0.0,0
2,0.222222,0.000000,0.000000,0.000000,0.111111,0.111111,0.222222,0.000000,0.0,0
3,0.555556,0.777778,0.777778,0.000000,0.222222,0.333333,0.222222,0.666667,0.0,0
4,0.333333,0.000000,0.000000,0.222222,0.111111,0.000000,0.222222,0.000000,0.0,0
